## Diving into the details of the inclusion implementations.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
from quickrules.data_handling import get_dataset
from pathlib import Path
import fuzzyroughrules.fuzzy_operators as fo
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from fuzzyroughrules.rule_induction_base import RuleGenerator
from fuzzyroughrules.approximations import LowerApproximation, MulticlassMSECVXApproximation
from fuzzyroughrules.operators import ImplicatorInclusion, OWAImplicatorInclusion
from quickrules.data_handling import test_save
from pathlib import Path
from sklearn.metrics import balanced_accuracy_score

In [16]:
name = 'ecoli' # 'bupa
nr = 6

In [17]:
x_train, y_train, t_train = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tra",
    get_datatypes=True
)
x_test, y_test = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tst"
)

In [18]:
classes = list(np.unique(np.append(y_train, y_test)))
y_train = np.array([classes.index(label) for label in y_train])
y_test = np.array([classes.index(label) for label in y_test])

In [19]:
model = RuleGenerator(
    LowerApproximation(),
    inclusion_measure=ImplicatorInclusion(),
    inclusion_threshold=0.9999,
    with_reducts=True,
)

In [20]:
model.fit(x_train, y_train, _)

In [21]:
model.predict(x_test)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 1, 1, 1, 1, 1, 0, 1, 1,
       4, 1, 4, 4, 5, 5, 7, 7, 5, 4, 7, 7])

In [22]:
model.selected_indexes

array([ 24,  25,  30,  36,  57,  59,  61,  86, 107, 130, 131, 132, 141,
       148, 161, 162, 166, 167, 171, 175, 188, 191, 194, 195, 197, 198,
       199, 200, 201, 210, 213, 214, 217, 221, 222, 223, 224, 226, 231,
       235, 236, 243, 244, 246, 251, 256, 257, 262, 264, 265, 268, 272,
       274, 285, 295, 299])

In [23]:
balanced_accuracy_score(y_test, model.predict(x_test))

0.8440476190476192

In [13]:
x_test_scaled = model.scaler.transform(x_test)

In [14]:
from fuzzyroughrules.operators import triangular_relation

In [15]:
credibility_predictions = []
for i in range(model.n_classes):
    credibility_predictions.append([])
for i in model.selected_indexes:
    credibility_predictions[model.y[i]].append(
        fo.lukasiewicz_t_norm(
            triangular_relation(
                x_test_scaled,
                model.X_scaled[i],
                model.slopes[i],
                model.reducts[i]
            ),
            model.positive_region[i]
        ).reshape((len(x_test_scaled)))
    )

In [16]:
credibility_predictions[7][0].shape

(34,)

In [17]:
model.n_classes, len(y_test)

(8, 34)

In [18]:
cumulative_credibility = np.zeros((x_test_scaled.shape[0], model.n_classes))
for i in range(model.n_classes):
    # print(len(credibility_predictions[i]))
    print(i)
    if len(credibility_predictions[i]) != 0:
        cumulative_credibility[:, i] = np.max(np.array(credibility_predictions[i]), 0) 

0
1
2
3
4
5
6
7


In [19]:
cumulative_credibility

array([[0.01260, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.18469, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.14391, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.02928, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.10351, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.09341, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.04290, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.10753, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.05010, 0.00000,
        0.03371],
       [0.27570, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.05747, 0.01601, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.16196, 0.00

In [20]:
np.argmax(model.predict_proba(x_test_scaled[3, :].reshape(1, -1), normalized=False), 1)

array([2])

In [24]:
balanced_accuracy_score(y_test, cumulative_credibility.argmax(axis=1))

NameError: name 'cumulative_credibility' is not defined

Comparing this to acc when using a higher threshold

In [25]:
from quickrules.weights import Weights, LinearWeights, TruncatedWeights

model_high = RuleGenerator(
    LowerApproximation(),
    # inclusion_measure=OWAImplicatorInclusion(
    #     weight_function=
    #         LinearWeights()
    # ),
    inclusion_measure=ImplicatorInclusion(),
    inclusion_threshold=1-1e-6,
    with_reducts=True,
    optimise_attribute_order=True
)
model_high.fit(x_train, y_train, t_train)
balanced_accuracy_score(y_test, model_high.predict(x_test))

0.8273809523809523

In [23]:
high_credibility_predictions = []
for i in range(model_high.n_classes):
    high_credibility_predictions.append([])
for i in model_high.selected_indexes:
    high_credibility_predictions[model_high.y[i]].append(
        fo.lukasiewicz_t_norm(
            triangular_relation(
                x_test_scaled,
                model_high.X_scaled[i],
                model_high.slopes[i],
                model_high.reducts[i]
            ),
            model_high.positive_region[i]
        )
    )

In [24]:
high_credibility_predictions[6][0]

array([0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
       0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
       0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
       0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
       0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000])

In [25]:
for i in range(model_high.n_classes):
    print(len(high_credibility_predictions[i]))

7
17
2
2
9
5
1
9
